# M5L1 - Genome Alignments in RNA-Seq and a Review of File Formats

In this Notebook, we will cover how genome aligners, like `STAR`, function, as well as the inputs and outputs to a STAR alignment, including a review of file formats used in alignment to a genome. 

**Learning Objectives:** 
- A brief review of File formats used in RNA-Seq analysis
- Understanding how alignment tools, like STAR, align reads to a reference genome

Expect this reading and review to take about 20 minutes.

```{admonition} Adapted from
:class: tip
**Source:** "Alignment with STAR" section adapted from [Introduction to RNA-seq using high-performance computing](https://hbctraining.github.io/Intro-to-rnaseq-hpc-O2/), Harvard Chan Bioinformatics Core (HBC), [hbctraining/Intro-to-rnaseq-hpc-O2](https://github.com/hbctraining/Intro-to-rnaseq-hpc-O2).

**License:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

**Changes:** The STAR explanation and figures were kept with light edits; the rest of the notebook (alignment types, file formats) is original; it is applied to Arabidopsis TAIR10 instead of human.
```

## Brief review of RNA-Seq, Genome Mapping, and file formats: 
*[Third-party RNA-seq workflow figure omitted from the public version. A similar workflow figure is in the [HBC Intro-to-DGE lessons](https://hbctraining.github.io/Intro-to-DGE/).]*

RNA-Seq is used to quantify gene expression. It measures the quantity and sequences of RNA within a sample, providing a snapshot of gene expression. RNA-Seq is used to identify which genes are activated, quantify gene expression, and detect alternative splicing or transcript variants. In the lab, RNA-Seq is used to compare gene expression across treatments, tissues, and time points to understand gene regulation within the samples.

---
The main steps involved in the processing of RNA-Seq Data are:

Step 1: **Pre-processing**
- Inputs: Raw sequencing reads (`fastq` files, usually paired-end)
- Steps: Quality check & Adapter trimming (`fastp`)
- Outputs: Trimmed and QC checked `fastq` files

Step 2: **Alignment / Mapping**
- Inputs: Trimmed and QC checked `fastq` files, Reference Genome/Transcriptome `fasta` file, and genome annotation (`gtf` or `gff3`
- Align reads to a reference genome or transcriptome (Genome alignment: `HISAT2`, `STAR` / Pseudo-Alignment: `Salmon`, `Kallisto`)
- Outputs: `BAM` files containing aligned reads

Step 3: **Annotation**
- Inputs: Aligned reads in `BAM` file format, gene annotation file (`GTF`/`GFF3`)
- Steps: Quality Assessment (`qualimap`), then Assign reads to genes/transcripts & generate count matrices for downstream analysis (`featureCounts`)
- Output: Count matrices, from raw or normalized counts (TPM or FKPM), are produced in `CSV`, `TSV`, or `TXT` format. 

Step 4: **Post-Processing**

After alignment and annotation, post-processing allows researchers to interpret RNA-Seq data biologically. It typically includes the following components:

- **Inputs**
    - Count matrix (either raw or normalized into TPM or FKPM)
    - Sample metadata (treatments, conditions, replicates, sequencing lanes, etc.)

- **Quality Assessment**
    - Verify that the alignment and annotation steps were successful.
    - Check for potential biases, batch effects, or sample outliers.
    - Tools like `MultiQC` can summarize quality metrics across samples.

- **Visualization**
    - PCA (Principal Component Analysis): reduces dimensionality to visualize sample relationships and batch effects.
    - Volcano plots: display differential expression, showing significance versus fold-change for genes.

- **Differential Expression Analysis** (`DESeq2`, `edgeR`)
    - Identify genes whose expression changes significantly between conditions or treatments.
    - Outputs include: List of DE genes in log fold-change, p-values, and adjusted p-values for multiple testing.
    - Heatmaps: *Done after DE generally* Visualize expression levels of selected genes across samples (VST) or across treatment groups (log2FC), highlighting patterns and clusters.

- **Functional Enrichment Analysis** (`clusterProfiler`)
    - Determine if sets of DE genes are overrepresented in particular biological pathways or processes.
    - Gene Ontology (GO) analysis identifies biological processes, molecular functions, and cellular components in reference to known genes.
    - KEGG pathway analysis highlights affected metabolic or signaling pathways.

---

### **Types of alignments used in RNA-Seq:** De-Novo, Genome, Transcriptome, Pseudo-alignment(to transcriptome)

For this week's labs, we are going to focus solely on alignment and visualization. The following alignment strategies exist within RNA-Seq processing: 

#### **1. De Novo Alignment**
- No reference genome required.  
- Reads are assembled into contigs → transcripts.  
- Useful for non-model organisms.  
- Computationally intensive; limited precision.
- Reading: [De novo assembly of transcriptomes and differential gene expression analysis using short-read data from emerging model organisms – a brief guide](https://frontiersinzoology.biomedcentral.com/articles/10.1186/s12983-024-00538-y)
![12983_2024_538_Fig2_HTML.png](images/M5L1_Genome_Alignments_and_File_Formats_34d09c6d.png)

*Image: Jackson, Cerveau & Posnien (2024) Front. Zool. 21:17, Fig. 2, CC BY 4.0, [doi:10.1186/s12983-024-00538-y](https://doi.org/10.1186/s12983-024-00538-y).*

#### **2. Genome Alignment**
- Reads aligned to a reference **genome (FASTA)**.  
- Detects spliced reads, novel junctions, and structural variation.  
- Necessary for splicing analysis and variant calling.  
- Tools: **STAR**, **HISAT2**, **Bowtie2** (unspliced).

#### **3. Transcriptome Alignment**
- Reads aligned to **known transcripts** instead of the genome.  
- Faster than genome alignment.  
- Cannot detect novel transcripts or novel splice junctions.  
- Used for expression quantification when annotation is complete.

#### **4. Pseudo-alignment (to the Transcriptome)**
- Assigns reads to **compatible transcripts** without full alignment.  
- Extremely fast and memory-efficient.  
- Ideal for quantification-only workflows.  
- Cannot detect novel splicing or variants.  
- Tools: **Kallisto**, **Salmon**.

---
### **File types used in genome alignments:**
*[Third-party figure omitted from the public version; original source not confirmed.]*

Traditionally, genome alignment is used to map reads individually to a reference genome. Genome alignment tools, such as `STAR` or `HISTAT2`, are very accurate. They provide genome alignments with specific coordinates and alignment scores for each read, but this process is very computationally intensive. These genome alignments can be viewed in genome browsers and used for variant calling, splice detection, or the discovery of new transcripts. To quantify the reads, additional tools must be used to produce count matrices for downstream differential expression analysis (`featureCounts`). 

**A genome alignment generally needs the following inputs:**
- A `fasta` file of the organism's genome 
- A `gff3` or `GTF` file containing the annotated genomic features of the reference genome
- Sequenced reads contained in `fastq` file format from each sample

Any aligner will need to build an **index**, or "look-up" map of all possible positions for alignment within the genome. 

*[Third-party figure omitted from the public version; original source not confirmed.]*

**The aligner will then output the following:**
- Aligned output files: `sam` or `bam` format
    - `STAR` can output: `Aligned.sortedByCoord.out.bam`, a coordinate-sorted BAM file ready for IGV or `Aligned.toTranscriptome.out.bam` (only if `--quantMode TranscriptomeSAM` is used and the star index is built with the transcriptome), a BAM with reads aligned to transcript coordinates (used by RSEM, Kallisto validation, etc.)
- Gene / Splice Junction Discovery Files: `SJ.out.tab`, which is a `tsv` table of splice junctions STAR detected during alignment.
- Log files used for QC checks: `Log.out`, `Log.progress.out`, `Log.final.out`, etc

# How does `STAR` work? 

The alignment process consists of choosing an appropriate reference genome to map our reads against and performing the read alignment using one of several splice-aware alignment tools such as STAR or HISAT2. The choice of aligner is often a personal preference and also dependent on the computational resources that are available to you.

**`STAR` Aligner**

To determine where on the human genome our reads originated from, we will align our reads to the reference genome using STAR (Spliced Transcripts Alignment to a Reference). [`STAR` (Spliced Transcripts Alignment to a Reference)](http://www.ncbi.nlm.nih.gov/pmc/articles/PMC3530905/ is an aligner designed to specifically address many of the challenges of RNA-seq data mapping using a strategy to account for spliced alignments.

### STAR Alignment Strategy

STAR is shown to have high accuracy and outperforms other aligners by more than a factor of 50 in mapping speed, but it is memory intensive. The algorithm achieves this highly efficient mapping by performing a two-step process:

    1. Seed searching
    2. Clustering, stitching, and scoring

#### 1. **Seed searching**

For every read that STAR aligns, STAR will search for the longest sequence that exactly matches one or more locations on the reference genome. These longest matching sequences are called the Maximal Mappable Prefixes (MMPs):

![alignment_STAR_step1.png](images/M5L1_Genome_Alignments_and_File_Formats_9d3841f5.png)

*Images in this section: Harvard Chan Bioinformatics Core (HBC), [Intro-to-rnaseq-hpc-O2](https://hbctraining.github.io/Intro-to-rnaseq-hpc-O2/), CC BY 4.0.*

The different parts of the read that are mapped separately are called ‘seeds’. So the first MMP that is mapped to the genome is called seed1.

STAR will then search again for only the unmapped portion of the read to find the next longest sequence that exactly matches the reference genome, or the next MMP, which will be seed2.

![alignment_STAR_step2.png](images/M5L1_Genome_Alignments_and_File_Formats_fadb9dec.png)

This sequential searching of only the unmapped portions of reads underlies the efficiency of the STAR algorithm. STAR uses an uncompressed suffix array (SA) to efficiently search for the MMPs, this allows for quick searching against even the largest reference genomes. Other slower aligners use algorithms that often search for the entire read sequence before splitting reads and performing iterative rounds of mapping.

**If STAR does not find an exact matching sequence** for each part of the read due to mismatches or indels, the previous MMPs will be extended.

![alignment_STAR_step3.png](images/M5L1_Genome_Alignments_and_File_Formats_094b77ea.png)

**If extension does not give a good alignment**, then the poor quality or adapter sequence (or other contaminating sequence) will be soft clipped.

![alignment_STAR_step4.png](images/M5L1_Genome_Alignments_and_File_Formats_8c149618.png)

#### 2. **Clustering, stitching, and scoring**

The separate seeds are stitched together to create a complete read by first clustering the seeds together based on proximity to a set of ‘anchor’ seeds, or seeds that are not multi-mapping.

Then the seeds are stitched together based on the best alignment for the read (scoring based on mismatches, indels, gaps, etc.).

![alignment_STAR_step5.png](images/M5L1_Genome_Alignments_and_File_Formats_53304cb2.png)

<iframe width="560" height="315" src="https://www.youtube.com/embed/3mJug4vfVAU?si=nbVzX81mU9yeXZjh" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>